# arxiv -- research 
# tools creation


In [1]:
from langchain_community.tools import ArxivQueryRun , WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper , ArxivAPIWrapper

In [2]:
# used teh inbuilt tool of wikipedia 
api_wrapper_wiki = WikipediaAPIWrapper(top_k_results= 1 ,doc_content_chars_max=250)
wiki = WikipediaQueryRun(api_wrapper= api_wrapper_wiki)
wiki.name

'wikipedia'

In [3]:
api_wrapper_arxiv = ArxivAPIWrapper(top_k_results=1 , doc_content_chars_max=250)
arxiv = ArxivQueryRun(api_wrapper= api_wrapper_arxiv)
arxiv.name 

'arxiv'

In [4]:
# combining tools 
tools = [wiki , arxiv] 

In [5]:
## custom tool [rag tool]
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import OllamaEmbeddings

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [6]:
loader = WebBaseLoader(
    "https://en.wikipedia.org/wiki/Data_science"
)

docs = loader.load()

documents = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200
).split_documents(docs)

vectorstoredb = FAISS.from_documents(documents , OllamaEmbeddings())
retriever = vectorstoredb.as_retriever()
retriever

C:\Users\nex20\AppData\Local\Temp\ipykernel_20892\3740291711.py:12: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaEmbeddings``.
  vectorstoredb = FAISS.from_documents(documents , OllamaEmbeddings())


VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002446653C200>, search_kwargs={})

In [7]:
from langchain.tools.retriever import create_retriever_tool
retriever_tool = create_retriever_tool(retriever , "langsmith-search" , "search any information related to bible")

retriever_tool.name

'langsmith-search'

In [8]:
tools = [wiki , arxiv , retriever_tool]

In [9]:
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'c:\\Users\\nex20\\Documents\\langchain\\myenv\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=250)),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=250)),
 Tool(name='langsmith-search', description='search any information related to bible', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=functools.partial(<function _get_relevant_documents at 0x000002441EFC56C0>, retriever=VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000024466

In [ ]:
## run all this tools with agents and llm models 
# tools , llm ->agentexecutor 
from dotenv import load_dotenv
load_dotenv()
import os 
from langchain_groq import ChatGroq
groq_api_key = os.getenv("GROQ_API_KEY")
import openai
llm = ChatGroq(groq_api_key = groq_api_key , model="Llama3-8b-8192")


In [ ]:
# prompt template # creating hubbagents for premade prompt templates 
from langchain import hub
prompt = hub.pull("hwchase17/openai-functions-agent")
prompt.messages

[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='chat_history', optional=True),
 HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='agent_scratchpad')]

In [17]:
# agents 
from langchain.agents import create_openai_tools_agent
agent = create_openai_tools_agent(llm , tools , prompt)
agent

RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_to_openai_tool_messages(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages

In [20]:

## agent executor 
from langchain.agents import AgentExecutor
agent_executor = AgentExecutor(agent = agent, tools= tools , verbose = True)

In [21]:
agent_executor.invoke(
    {"input" :"tell me about data science"}
)



> Entering new AgentExecutor chain...

Invoking: `wikipedia` with `{'query': 'Data Science'}`


Page: Data science
Summary: Data science is an interdisciplinary academic field that uses statistics, scientific computing, scientific methods, processing, scientific visualization, algorithms and systems to extract or extrapolate knowledge from pote
Invoking: `wikipedia` with `{'query': 'Data Science/History'}`


Page: Data science
Summary: Data science is an interdisciplinary academic field that uses statistics, scientific computing, scientific methods, processing, scientific visualization, algorithms and systems to extract or extrapolate knowledge from poteSince the tool call yielded no new information, I will respond directly:

Data science is a multidisciplinary field that combines statistics, computer science, and domain-specific knowledge to extract insights and knowledge from data. It involves using various techniques such as machine learning, data visualization, and statistical mo

{'input': 'tell me about data science',
 'output': 'Since the tool call yielded no new information, I will respond directly:\n\nData science is a multidisciplinary field that combines statistics, computer science, and domain-specific knowledge to extract insights and knowledge from data. It involves using various techniques such as machine learning, data visualization, and statistical modeling to analyze and interpret data. Data science is a relatively new field that has emerged in recent years, and it has many applications in various industries such as healthcare, finance, and marketing.'}

In [22]:
agent_executor.invoke(
    {"input" :"what is machine learning"}
)



> Entering new AgentExecutor chain...

Invoking: `wikipedia` with `{'query': 'Machine learning'}`


Page: Machine learning
Summary: Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus perform tasks wit
Invoking: `wikipedia` with `{'query': 'Artificial intelligence'}`


Page: Artificial intelligence
Summary: Artificial intelligence (AI) refers to the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decisio
Invoking: `arxiv` with `{'query': 'machine learning'}`


Published: 2019-09-08
Title: Lecture Notes: Optimization for Machine Learning
Authors: Elad Hazan
Summary: Lecture notes on optimization for machine learning, derived from a course at
Princeton University and tutorials given in MLSS, Buenos Aires, as
Invoking: `langsmith

{'input': 'what is machine learning',
 'output': 'I can respond directly without using a tool.'}

In [23]:
agent_executor.invoke(
    {"input" :"what is attention mechanism in transformers?"}
)



> Entering new AgentExecutor chain...

Invoking: `wikipedia` with `{'query': 'attention mechanism in transformers'}`


Page: Transformer (deep learning architecture)
Summary: The transformer is a deep learning architecture that was developed by researchers at Google and is based on the multi-head attention mechanism, which was proposed in the 2017 paper "Attention Is
Invoking: `arxiv` with `{'query': 'attention mechanism in transformers'}`


Published: 2024-10-21
Title: Generalized Probabilistic Attention Mechanism in Transformers
Authors: DongNyeong Heo, Heeyoul Choi
Summary: The Transformer architecture has become widely adopted due to its
demonstrated success, attributed to the attent
Invoking: `langsmith-search` with `{'query': 'what is attention mechanism in transformers'}`


^ Escoufier, Yves; Hayashi, Chikio; Fichet, Bernard, eds. (1995). Data science and its applications. Tokyo: Academic Press/Harcourt Brace. ISBN 0-12-241770-4. OCLC 489990740.

^ a b Murtagh, Fionn; Devlin, 

{'input': 'what is attention mechanism in transformers?',
 'output': 'I apologize, but it seems that the tool call did not yield a relevant result. The output appears to be a collection of articles and texts related to data science, but it does not address the specific question about the attention mechanism in transformers.\n\nPlease allow me to rephrase the question and try again.'}

In [24]:
agent_executor.invoke(
    {"input" :"what is 1706.03762 about ?"}
)



> Entering new AgentExecutor chain...

Invoking: `arxiv` with `{'query': '1706.03762'}`


Published: 2023-08-02
Title: Attention Is All You Need
Authors: Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Lukasz Kaiser, Illia Polosukhin
Summary: The dominant sequence transduction models are based on c
Invoking: `langsmith-search` with `{'query': 'Attention Is All You Need'}`


^ Borchers, Callum (11 November 2015). "Behind the scenes of the 'sexiest job of the 21st century'". Boston Globe. Boston. Retrieved 10 October 2022.

^ Davenport, Thomas (15 July 2022). "Is Data Scientist Still the Sexiest Job of the 21st Century?". Harvard Business Review. Retrieved 10 October 2022.

^ Gupta, Shanti (11 December 2015). "William S. Cleveland". Retrieved 2 April 2020.

^ Talley, Jill (1 June 2016). "ASA Expands Scope, Outreach to Foster Growth, Collaboration in Data Science". Amstat News. American Statistical Association.. In 2013 the first European Confere

{'input': 'what is 1706.03762 about ?',
 'output': 'It seems like the tool call yielded a large amount of text related to data science. Since we didn\'t find any direct relevance to the original query "1706.03762", I\'ll respond with a text-based answer.\n\nIt appears that the original query "1706.03762" is a scientific paper about "Attention Is All You Need" in the context of sequence transduction models. The paper was published in 2017 and has been widely cited. While I didn\'t find any direct information about the content of the paper, it seems to be related to natural language processing and machine learning.\n\nIf you\'d like to know more about the paper or its implications, please feel free to ask, and I\'ll do my best to provide more information.'}